In [1]:
from transformers import GPT2LMHeadModel
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

torch.set_default_device(device='cuda')
torch.manual_seed(1234)

c:\Users\toprak\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class CausalSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))
    def forward(self,x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2) 
        # n_embd x (3 * n_embd) boyutlu tek c_attn katmanini uce ayirir. boylece q,k,v matrislerinin hepsi ayri bir girdiymis gibi, batch icindeki elemanlar gibi hesaplanir.
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1,2) 
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        # q,k,v'nin icindeki headler ayrilir, headler birbirlerinden izole edilerek batch dimension haline getilir. dort boyutlu tensorlar olusmus olur
        # transpoze edilmesinin sebebi de her headin ayri bir batch dimension olarak alinmasi, T yani ornekler boyutu ile C yani kanallar her head icinde bagimsiz isleme alinmalidir.
        attn = q @ k.transpose(-2,-1)
        attn = attn * torch.sqrt(torch.tensor(self.n_embd//self.n_head))**-1
        attn = attn.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        out = attn @ v
        out = out.transpose(1,2).contiguous().view(B, T, C)
        # hesaplamalar yapildiktan sonra tekrar eski haline getiriliyor
        out = self.c_proj(out)
        return out

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config=config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config=config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size:int = 1024
    vocab_size:int = 50257
    n_layers:int = 12
    n_head:int = 12
    n_embd:int = 768

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(wte = nn.Embedding(config.vocab_size, config.n_embd), wpe = nn.Embedding(config.block_size, config.n_embd), h = nn.ModuleList([Block(config) for _ in range(config.n_layers)]), ln_f = nn.LayerNorm(config.n_embd)))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    @classmethod
    def from_pretrained(cls, model_type):
        configs = {
            'gpt2': dict(n_layers = 12, n_head = 12, n_embd = 768),
            'gpt2-medium': dict(n_layers = 24, n_head = 16, n_embd = 1024),
            'gpt2-large': dict(n_layers = 36, n_head = 20, n_embd = 1280),
            'gpt2-xl': dict(n_layers = 48, n_head = 25, n_embd = 1600)
        }[model_type]
        configs['vocab_size'] = 50257
        configs['block_size'] = 1024
        config = GPTConfig(**configs)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = [k for k in sd.keys() if not k.endswith('.attn.masked_bias')]

        hf_model = GPT2LMHeadModel.from_pretrained(model_type)
        hf_sd = hf_model.state_dict()
        hf_sd_keys = [k for k in hf_sd.keys() if not k.endswith('.attn.masked_bias') and not k.endswith('.attn.bias')]
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']

        for k in hf_sd_keys:
            if any(k.endswith(w) for w in transposed):
                assert hf_sd[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(hf_sd[k].t())
            else:
                assert hf_sd[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(hf_sd[k])
        return model
    def forward(self, x):
        token_embeddings = self.transformer.wte(x)
        positional_embeddings = self.transformer.wpe(torch.arange(x.shape[1]))
        x = token_embeddings + positional_embeddings
        for block in self.transformer['h']:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        return logits

In [3]:
model = GPT.from_pretrained('gpt2')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 394.69it/s]


In [5]:
import tiktoken

model = GPT.from_pretrained('gpt2')
model_hf = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()
encoder = tiktoken.get_encoding("gpt2")
text = "Deneme 123"
tokens = encoder.encode(text)
x = torch.tensor(tokens, dtype=torch.long).unsqueeze(0)
with torch.no_grad():
    logits = model(x)
model_hf.eval()
logits_hf = model_hf(x).logits

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 407.18it/s]


In [6]:
logits_hf

tensor([[[-28.8699, -28.2894, -31.1253,  ..., -36.7023, -35.9628, -28.5520],
         [-86.9684, -85.7624, -92.8295,  ..., -96.3223, -93.9189, -88.4780],
         [-82.1657, -80.6922, -81.4508,  ..., -91.8775, -89.6322, -80.2975]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)

In [7]:
logits

tensor([[[-28.8699, -28.2894, -31.1253,  ..., -36.7023, -35.9627, -28.5519],
         [-86.9684, -85.7624, -92.8295,  ..., -96.3223, -93.9189, -88.4780],
         [-82.1657, -80.6922, -81.4508,  ..., -91.8775, -89.6322, -80.2975]]],
       device='cuda:0')

In [ ]:
torch.allclose(logits, logits_hf)

True

: 